In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import seaborn as sns
import cv2
from PIL import Image

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("OpenCV:", cv2.__version__)
print("Semua library berhasil diimpor!")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent

DATASET_DIR = PROJECT_DIR / "datasets" / "dataset-pabrik-kalsel"

SPLITS = {
    "train": DATASET_DIR / "train_rev2" / "train",
    "valid": DATASET_DIR / "valid_rev2" / "valid",
    "test": DATASET_DIR / "test_rev2" / "test",
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

results = []

for split_name, split_dir in SPLITS.items():

    if not split_dir.exists():
        print(f"Folder tidak ditemukan: {split_dir}")
        continue

    images = [
        file for file in split_dir.rglob("*")
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    ]

    annotations = [
        file for file in split_dir.rglob("*.txt")
    ]

    annotations = [
        file for file in annotations
        if file.name.lower() != "_darknet.labels"
    ]

    total_objects = 0
    empty_annotations = 0

    for annotation in annotations:
        lines = [
            line.strip()
            for line in annotation.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

        total_objects += len(lines)

        if len(lines) == 0:
            empty_annotations += 1

    results.append({
        "Split": split_name,
        "Jumlah Gambar": len(images),
        "Jumlah File Anotasi": len(annotations),
        "Total Objek": total_objects,
        "Anotasi Kosong": empty_annotations
    })

eda_summary = pd.DataFrame(results)

display(eda_summary)

print("\nLokasi dataset:", DATASET_DIR.resolve())

In [ ]:
from collections import Counter

class_names = [
    "Janjang kosong",
    "Kurang masak",
    "TBS abnormal",
    "TBS masak",
    "TBS mentah",
    "Terlalu masak"
]

class_counts = Counter()

for split_name, split_dir in SPLITS.items():
    for annotation in split_dir.rglob("*.txt"):

        if annotation.name.lower() == "_darknet.labels":
            continue

        lines = annotation.read_text(
            encoding="utf-8"
        ).splitlines()

        for line in lines:
            if not line.strip():
                continue

            values = line.split()
            class_id = int(values[0])

            class_counts[class_id] += 1

# Susun hasil distribusi kelas
class_distribution = pd.DataFrame([
    {
        "Class ID": class_id,
        "Nama Kelas": class_names[class_id],
        "Jumlah Objek": class_counts[class_id]
    }
    for class_id in range(len(class_names))
])

class_distribution["Persentase (%)"] = (
    class_distribution["Jumlah Objek"]
    / class_distribution["Jumlah Objek"].sum()
    * 100
).round(2)

display(class_distribution)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))

sns.barplot(
    data=class_distribution,
    x="Nama Kelas",
    y="Jumlah Objek"
)

plt.title("Distribusi Kelas Dataset Pabrik Kalimantan Selatan")
plt.xlabel("Kelas TBS")
plt.ylabel("Jumlah Objek")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
from collections import Counter

split_class_counts = []

for split_name, split_dir in SPLITS.items():
    counts = Counter()

    for annotation in split_dir.rglob("*.txt"):
        if annotation.name.lower() == "_darknet.labels":
            continue

        for line in annotation.read_text(
            encoding="utf-8"
        ).splitlines():

            if not line.strip():
                continue

            class_id = int(line.split()[0])
            counts[class_id] += 1

    for class_id, class_name in enumerate(class_names):
        split_class_counts.append({
            "Split": split_name,
            "Class ID": class_id,
            "Nama Kelas": class_name,
            "Jumlah Objek": counts[class_id]
        })

split_distribution = pd.DataFrame(split_class_counts)

# Tampilkan dalam bentuk tabel per kelas
split_pivot = split_distribution.pivot(
    index="Nama Kelas",
    columns="Split",
    values="Jumlah Objek"
)

display(split_pivot)


In [ ]:
plt.figure(figsize=(11, 6))

sns.barplot(
    data=split_distribution,
    x="Nama Kelas",
    y="Jumlah Objek",
    hue="Split"
)

plt.title("Distribusi Kelas Berdasarkan Dataset Split")
plt.xlabel("Kelas TBS")
plt.ylabel("Jumlah Objek")
plt.xticks(rotation=30, ha="right")
plt.legend(title="Dataset Split")
plt.tight_layout()
plt.show()


In [ ]:

validation_results = []

for split_name, split_dir in SPLITS.items():

    total_boxes = 0
    invalid_format = 0
    invalid_class = 0
    invalid_coordinates = 0
    invalid_size = 0

    for annotation in split_dir.rglob("*.txt"):

        if annotation.name.lower() == "_darknet.labels":
            continue

        for line_number, line in enumerate(
            annotation.read_text(encoding="utf-8").splitlines(),
            start=1
        ):

            if not line.strip():
                continue

            total_boxes += 1
            values = line.split()

            if len(values) != 5:
                invalid_format += 1
                continue

            try:
                class_id = int(values[0])
                x_center, y_center, width, height = map(
                    float, values[1:]
                )
            except ValueError:
                invalid_format += 1
                continue

            if class_id not in range(6):
                invalid_class += 1

            if not all(
                0 <= value <= 1
                for value in [x_center, y_center, width, height]
            ):
                invalid_coordinates += 1

            if width <= 0 or height <= 0:
                invalid_size += 1

    validation_results.append({
        "Split": split_name,
        "Total Bounding Box": total_boxes,
        "Format Tidak Valid": invalid_format,
        "Class ID Tidak Valid": invalid_class,
        "Koordinat Tidak Valid": invalid_coordinates,
        "Ukuran Tidak Valid": invalid_size
    })

bbox_validation = pd.DataFrame(validation_results)

display(bbox_validation)


In [ ]:
boundary_results = []

for split_name, split_dir in SPLITS.items():

    total_boxes = 0
    out_of_bounds = 0

    for annotation in split_dir.rglob("*.txt"):

        if annotation.name.lower() == "_darknet.labels":
            continue

        for line in annotation.read_text(
            encoding="utf-8"
        ).splitlines():

            if not line.strip():
                continue

            values = line.split()

            if len(values) != 5:
                continue

            _, xc, yc, w, h = map(float, values)

            total_boxes += 1

            # Hitung batas bounding box
            x_min = xc - w / 2
            x_max = xc + w / 2
            y_min = yc - h / 2
            y_max = yc + h / 2

            # Periksa apakah kotak melewati batas gambar
            if (
                x_min < 0 or x_max > 1
                or y_min < 0 or y_max > 1
            ):
                out_of_bounds += 1

    boundary_results.append({
        "Split": split_name,
        "Total Bounding Box": total_boxes,
        "Bounding Box Keluar Batas": out_of_bounds
    })

boundary_df = pd.DataFrame(boundary_results)

display(boundary_df)


In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Supaya sampel yang dipilih konsisten setiap kali dijalankan
random.seed(42)

# Nama kelas
class_names = [
    "Janjang kosong",
    "Kurang masak",
    "TBS abnormal",
    "TBS masak",
    "TBS mentah",
    "Terlalu masak"
]

# Folder train
train_dir = SPLITS["train"]

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Cari gambar yang memiliki file anotasi
image_files = [
    path for path in train_dir.rglob("*")
    if path.is_file()
    and path.suffix.lower() in image_extensions
    and path.with_suffix(".txt").exists()
]

# Pilih maksimal 6 gambar
sample_images = random.sample(
    image_files,
    min(6, len(image_files))
)

# Visualisasikan gambar dan bounding box
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for ax, image_path in zip(axes, sample_images):

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width = image.shape[:2]

    annotation_path = image_path.with_suffix(".txt")

    for line in annotation_path.read_text(
        encoding="utf-8"
    ).splitlines():

        if not line.strip():
            continue

        class_id, xc, yc, bw, bh = map(float, line.split())

        # Konversi koordinat YOLO ke pixel
        x1 = int((xc - bw / 2) * width)
        y1 = int((yc - bh / 2) * height)
        x2 = int((xc + bw / 2) * width)
        y2 = int((yc + bh / 2) * height)

        # Gambar bounding box
        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        # Tampilkan nama kelas
        label = class_names[int(class_id)]

        cv2.putText(
            image,
            label,
            (x1, max(y1 - 8, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 0, 0),
            2
        )

    ax.imshow(image)
    ax.set_title(image_path.name, fontsize=9)
    ax.axis("off")

# Sembunyikan subplot yang tidak terpakai
for ax in axes[len(sample_images):]:
    ax.axis("off")

plt.suptitle(
    "Visualisasi Bounding Box Dataset Pabrik - Train",
    fontsize=16
)

plt.tight_layout()
plt.show()


In [ ]:
from PIL import Image
import pandas as pd

image_dimensions = []

for split_name, split_dir in SPLITS.items():

    for image_path in split_dir.rglob("*"):

        if (
            image_path.is_file()
            and image_path.suffix.lower()
            in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        ):

            try:
                with Image.open(image_path) as img:
                    width, height = img.size

                image_dimensions.append({
                    "Split": split_name,
                    "Nama File": image_path.name,
                    "Width": width,
                    "Height": height,
                    "Aspect Ratio": round(width / height, 3)
                })

            except Exception as e:
                print(f"Gagal membaca {image_path}: {e}")

dimensions_df = pd.DataFrame(image_dimensions)

print("Total gambar diperiksa:", len(dimensions_df))

display(
    dimensions_df.groupby("Split").agg(
        Jumlah_Gambar=("Nama File", "count"),
        Width_Min=("Width", "min"),
        Width_Max=("Width", "max"),
        Width_Rata_Rata=("Width", "mean"),
        Height_Min=("Height", "min"),
        Height_Max=("Height", "max"),
        Height_Rata_Rata=("Height", "mean")
    ).round(2)
)


In [ ]:
resolution_counts = (
    dimensions_df
    .groupby(["Width", "Height"])
    .size()
    .reset_index(name="Jumlah Gambar")
    .sort_values("Jumlah Gambar", ascending=False)
)

display(resolution_counts.head(15))

print(
    "Jumlah variasi resolusi:",
    len(resolution_counts)
)


In [ ]:
import cv2
import numpy as np
import pandas as pd

quality_results = []
corrupted_images = []

for split_name, split_dir in SPLITS.items():

    for image_path in split_dir.rglob("*"):

        if (
            not image_path.is_file()
            or image_path.suffix.lower()
            not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        ):
            continue

        image = cv2.imread(str(image_path))

        # Periksa apakah gambar berhasil dibaca
        if image is None:
            corrupted_images.append({
                "Split": split_name,
                "File": image_path.name
            })
            continue

        # Ubah ke grayscale untuk analisis kualitas
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Hitung karakteristik gambar
        brightness = np.mean(gray)
        contrast = np.std(gray)
        sharpness = cv2.Laplacian(
            gray, cv2.CV_64F
        ).var()

        quality_results.append({
            "Split": split_name,
            "Nama File": image_path.name,
            "Brightness": brightness,
            "Contrast": contrast,
            "Sharpness": sharpness
        })

quality_df = pd.DataFrame(quality_results)

print("Total gambar berhasil dibaca:", len(quality_df))
print("Total gambar rusak:", len(corrupted_images))

display(
    quality_df.groupby("Split").agg(
        Brightness_Rata_Rata=("Brightness", "mean"),
        Contrast_Rata_Rata=("Contrast", "mean"),
        Sharpness_Rata_Rata=("Sharpness", "mean")
    ).round(2)
)

if corrupted_images:
    display(pd.DataFrame(corrupted_images))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

metrics = [
    ("Brightness", "Distribusi Kecerahan"),
    ("Contrast", "Distribusi Kontras"),
    ("Sharpness", "Distribusi Ketajaman")
]

for ax, (column, title) in zip(axes, metrics):

    sns.histplot(
        data=quality_df,
        x=column,
        hue="Split",
        kde=True,
        ax=ax
    )

    ax.set_title(title)

plt.tight_layout()
plt.show()


In [ ]:
# Ambil 10 gambar dengan nilai sharpness terendah
lowest_sharpness = quality_df.nsmallest(10, "Sharpness")

display(
    lowest_sharpness[
        ["Split", "Nama File", "Brightness", "Contrast", "Sharpness"]
    ].round(2)
)

In [ ]:
import matplotlib.pyplot as plt
import cv2

sample_low = lowest_sharpness.head(6)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, (_, row) in zip(axes, sample_low.iterrows()):

    split_dir = SPLITS[row["Split"]]
    image_path = next(
        split_dir.rglob(row["Nama File"])
    )

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    ax.imshow(image)
    ax.set_title(
        f'{row["Nama File"][:25]}\nSharpness: {row["Sharpness"]:.2f}',
        fontsize=9
    )
    ax.axis("off")

plt.suptitle("Enam Gambar dengan Sharpness Terendah")
plt.tight_layout()
plt.show()

In [19]:
# Membuat folder penyimpanan hasil EDA
RESULTS_DIR = PROJECT_DIR / "results" / "eda_dataset_pabrik"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Simpan distribusi kelas
class_distribution.to_csv(
    RESULTS_DIR / "distribusi_kelas.csv",
    index=False
)

# Simpan distribusi kelas per split
split_distribution.to_csv(
    RESULTS_DIR / "distribusi_kelas_per_split.csv",
    index=False
)

# Simpan hasil dimensi gambar
dimensions_df.to_csv(
    RESULTS_DIR / "dimensi_gambar.csv",
    index=False
)

# Simpan hasil pemeriksaan kualitas gambar
quality_df.to_csv(
    RESULTS_DIR / "kualitas_gambar.csv",
    index=False
)

# Simpan hasil validasi bounding box
bbox_validation.to_csv(
    RESULTS_DIR / "validasi_bounding_box.csv",
    index=False
)

# Simpan pemeriksaan batas bounding box
boundary_df.to_csv(
    RESULTS_DIR / "validasi_batas_bounding_box.csv",
    index=False
)

print("Seluruh hasil EDA berhasil disimpan!")
print("Lokasi:", RESULTS_DIR)

Seluruh hasil EDA berhasil disimpan!
Lokasi: c:\Users\gabes\OneDrive\GABE_JUNTAK\SEMESTER 5\CERTAN\PRAKTIKUM\benchmark-tbs-rf-detr-vs-yolo\results\eda_dataset_pabrik
